In [1]:
import papermill as pm
import os
import subprocess
import datetime
from pylatex import Document, Section, Subsection, Command, NoEscape, Package
import scrapbook as sb
import pandas as pd


In [2]:

# Paths
date = datetime.datetime.now().strftime("%Y-%m-%d/%H-%M")
NOTEBOOK_PATHS = [
    # # Base Models
    '../diffusion_model/gesture_train.ipynb',
    '../diffusion_model/phase_train_from_scratch.ipynb',
    # '../diffusion_model/grs_train.ipynb',
    # Transfer Learning Models
    '../transfer_learning/phase_train_full_model_fine_tuning.ipynb',
    '../transfer_learning/phase_train_frozen_encoder.ipynb',
    '../transfer_learning/phase_train_linear_probe.ipynb',
    # Ablation Models
    '../ablation/phase_train_w_o_language.ipynb',
    '../ablation/phase_train_w_o_action.ipynb',
    '../ablation/phase_train_w_o_time.ipynb',
    '../ablation/phase_train_w_o_action_language.ipynb',
    '../ablation/phase_train_w_o_action_time.ipynb',
    '../ablation/phase_train_w_o_language_time.ipynb',
    '../ablation/phase_train_vision_only.ipynb'
    
]

TASKS = ['Suturing', 'Needle_Passing', 'Knot_Tying']
# TASKS = ['Suturing']
OUTPUT_DIR = f'../../outputs/executed_notebooks/experiments/{date}/'
REPORT_PATH = f'../../outputs/reports/{date}/report'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)


In [3]:
# Initialise Report
report = Document()
report.packages.append(Package('booktabs'))
report.preamble.append(Command('title', f'Experiment Report {date}'))
report.preamble.append(Command('date', f'{date}'))
report.append(NoEscape(r'\maketitle'))


In [4]:

# Initialize lists to store executed notebooks, metadata, and results
executed_notebooks = []
metadatas = []
results = []
training_records = []

# Initialize variables for hyperparameter testing
LRS = [1e-4]
BATCH_SIZES = [32]
HIDDEN_DIMS = [2048]
EPOCHS = [25]

In [5]:
# Function to generate output notebook path
def get_output_path(notebook, task, output_dir):
    notebook_name = os.path.basename(notebook)
    name = notebook_name.split('.')[0] + '_' + task
    return os.path.join(output_dir, name)


In [ ]:
# Execute notebooks for each task and save outputs
count = 0
total = len(TASKS) * len(LRS) * len(BATCH_SIZES) * len(HIDDEN_DIMS) * len(EPOCHS) * len(NOTEBOOK_PATHS)
   
for lr_idx, lr in enumerate(LRS):
    lr_val = len(str(lr).split('.')[-1])
    for batch_size in BATCH_SIZES:
        for hidden_dim in HIDDEN_DIMS:
            for epoch in EPOCHS:
                for task in TASKS: 
                    for notebook in NOTEBOOK_PATHS:
                        count += 1
                        output_path = get_output_path(notebook, task, OUTPUT_DIR)
                        output_notebook = output_path + f'_{lr_val}_{batch_size}_{hidden_dim}_{epoch}' + '.ipynb'

                        params = {'TASK': task, 'date': date, 'LR':lr, 'HIDDEN_DIM': hidden_dim, 'BATCH_SIZE': batch_size, 'EPOCHS': epoch}
                        print(f"Running experiment {count}/{total}...")
                        print(f"Executing {notebook} for task {task} with LR={lr}, BATCH_SIZE={batch_size}, HIDDEN_DIM={hidden_dim}...")
                        pm.execute_notebook(
                            notebook,
                            output_notebook,
                            parameters=params,
                            progress_bar=True
                        )

                        # Add executed notebook to the list
                        executed_notebooks.append(output_notebook)

                        try:
                            output_pdf_filename = os.path.basename(output_path) + f'_{lr_val}_{batch_size}_{hidden_dim}' + '.pdf'
                            subprocess.run([
                                'jupyter', 'nbconvert', '--to', 'pdf', output_notebook,
                                '--output', output_pdf_filename,
                                '--output-dir', OUTPUT_DIR
                            ], check=True)
                        except subprocess.CalledProcessError as e:
                            print(f"Error occurred while converting {output_notebook} to PDF: {e}")


Running experiment 1/36...
Executing ../diffusion_model/gesture_train.ipynb for task Suturing with LR=0.0001, BATCH_SIZE=32, HIDDEN_DIM=2048...


Executing:   0%|          | 0/17 [00:00<?, ?cell/s]

In [ ]:
# Extract metadata and results from executed notebooks
for nb in executed_notebooks:
    notebook = sb.read_notebook(nb)

    # extract metadata and results
    metadata = notebook.scraps['metadata'].data
    result = notebook.scraps['results'].data

    if isinstance(metadata, list):
        metadatas.extend(metadata)
    else:
        metadatas.append(metadata)

    if isinstance(result, list):
        results.extend(result)
    else:
        results.append(result)
        

In [ ]:
# Save metadata, training epochs records and results to report
for metadata, result in zip(metadatas, results):
    with report.create(Section(f"Experiment: {result['kind']} {result['task']} {result['type']}")):
        with report.create(Subsection("Metadata")):
            for key, value in metadata.items():
                report.append(f"{key}: {value}\n\n")
        with report.create(Subsection("Results")):
            for key, value in result.items():
                report.append(f"{key}: {value}\n\n")

In [ ]:
# Generate and save the report
report.generate_pdf(REPORT_PATH, clean_tex=True)


In [ ]:
# Generate a table of all results
df = pd.DataFrame(results)
df = df[['kind', 'task', 'type','accuracy', 'train_time', 'top5_accuracy', 'weighted_f1', 'lr', 'batch_size', 'hidden_dim']]


print(df.columns)


In [ ]:
df.to_csv(REPORT_PATH + '_results.csv', index=False)